Experiment 3 -- Computational overhead and per-request latency (RQ3)

In [1]:
# 03_latency_overhead.ipynb
#
# Experiment 3 -- Computational overhead and per-request latency (RQ3)
#
# Purpose:
#   Measure the added latency of the governance wrapper (model inference plus
#   group-wise calibration and an audit-log write) versus a bare model, to
#   support the real-time claim. Reports mean per-request latency over many
#   repetitions and the relative overhead.
#
# Outputs:
#   results/tables/exp3_latency.csv
#   results/figures/exp3_latency.(png|pdf)

# ==== Imports and grayscale academic style (600 dpi, PNG+PDF, no captions) ====
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
FIG_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
for d in (DATA_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)

sns.set_theme(style="whitegrid")
GRAYS = ["#000000", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYS))
plt.rcParams.update({
    "figure.dpi": 600, "savefig.dpi": 600, "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "grid.color": "0.85",
})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

def disparate_impact_ratio(y_pred, group):
    approve = (y_pred == 0).astype(int)
    r1 = approve[group == 1].mean(); r0 = approve[group == 0].mean()
    return min(r1, r0) / max(r1, r0) if max(r1, r0) > 0 else np.nan

def equalized_odds_gaps(y_true, y_pred, group):
    def rate(ct, mask):
        idx = (y_true == ct) & mask
        return (y_pred[idx] == 1).mean() if idx.sum() else np.nan
    a, b = group == 1, group == 0
    return abs(rate(1, a) - rate(1, b)), abs(rate(0, a) - rate(0, b))



from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
N = 8000
protected = rng.binomial(1, 0.35, N)
residential_region = 1.6 * protected + rng.normal(0, 0.6, N)
spending_pattern = 1.4 * protected + rng.normal(0, 0.6, N)
income = rng.normal(0, 1, N); debt_ratio = rng.normal(0, 1, N); pay_history = rng.normal(0, 1, N)
logit = (-0.9*income + 0.8*debt_ratio - 0.7*pay_history
         + 1.3*residential_region + 1.1*spending_pattern + rng.normal(0, 0.5, N))
default = (logit > np.quantile(logit, 0.7)).astype(int)
Xdf = pd.DataFrame(dict(residential_region=residential_region, spending_pattern=spending_pattern,
                        income=income, debt_ratio=debt_ratio, pay_history=pay_history))

Xtr, Xte, ytr, yte, gtr, gte = train_test_split(Xdf, default, protected, test_size=0.3, random_state=1)
model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)

# Calibrate group thresholds once (offline), as the wrapper would
p_tr = model.predict_proba(Xtr.values)[:, 1]
base = ((p_tr >= 0.5).astype(int) == 0).mean()
th = {0: np.quantile(p_tr[gtr == 0], base), 1: np.quantile(p_tr[gtr == 1], base)}

REPS = 5000
Xte_arr = Xte.values; gte_arr = np.asarray(gte)
idx = np.arange(REPS) % len(Xte_arr)

# Baseline: bare model inference per request
t0 = time.perf_counter()
for i in idx:
    _ = model.predict_proba(Xte_arr[i:i+1])
baseline_ms = (time.perf_counter() - t0) / REPS * 1000

# Wrapper: inference + group-threshold decision + audit-record append
audit = []
t0 = time.perf_counter()
for i in idx:
    pr = model.predict_proba(Xte_arr[i:i+1])[0, 1]
    g = int(gte_arr[i]); dec = int(pr >= th[g])
    audit.append({"req": int(i), "score": float(pr), "group": g, "decision": dec})
wrapper_ms = (time.perf_counter() - t0) / REPS * 1000

overhead_pct = (wrapper_ms / baseline_ms - 1) * 100
res = pd.DataFrame([
    {"pipeline": "baseline_model", "latency_ms_per_request": baseline_ms, "overhead_pct": 0.0},
    {"pipeline": "aigovalt_wrapper", "latency_ms_per_request": wrapper_ms, "overhead_pct": overhead_pct},
])
res.to_csv(os.path.join(TAB_DIR, "exp3_latency.csv"), index=False)
print(res.round(4).to_string(index=False))
print(f"Absolute added latency: {wrapper_ms - baseline_ms:.4f} ms/request")

fig, ax = plt.subplots(figsize=(5.2, 4.0))
ax.bar([0, 1], res["latency_ms_per_request"], edgecolor="black", linewidth=1.0,
       color=["#999999", "#000000"])
ax.set_xticks([0, 1]); ax.set_xticklabels(["Baseline model", "AI-Gov-Alt wrapper"])
ax.set_ylabel("Latency (ms per request)")
fig.tight_layout(); save_fig(fig, "exp3_latency"); plt.close(fig)
print("Saved latency figure and table.")


        pipeline  latency_ms_per_request  overhead_pct
  baseline_model                  0.0664        0.0000
aigovalt_wrapper                  0.0714        7.5033
Absolute added latency: 0.0050 ms/request


Saved latency figure and table.
